<a href="https://colab.research.google.com/github/Fatima-05/FlyRank-ML/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fatima-05/FlyRank-ML/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + methodology questions

Finding A — ranking / opportunity scoring can beat a transparent rule on a shortlist metric.
Question: where does the label come from, and does the validation design support a decision-support claim rather than a causal claim?
In this track the label is a proxy (declining trend), not confirmed editorial outcome. A client-holdout split supports "measured ranking quality on held-out clients," not "refreshing causes traffic."

Finding B — baseline rules are still useful as a fair comparison and as a simple production fallback.
Question: is the baseline evaluated on the same split and metric as the model?
It should be. If the rule uses the label itself, the comparison is invalid. Our baseline uses non-label signals only and is scored with the same Precision@50 on the same holdout.

In [5]:
print("Audit focus: proxy label source + same-split comparison + no causal overclaim")

Audit focus: proxy label source + same-split comparison + no causal overclaim


## 2. My model under an honest split

Before: a weak comparison would be a random row split, or a baseline that includes the label.
After: GroupShuffleSplit by client_id, baseline without trend_direction, Precision@50 on the same test rows.

In [6]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier

if not Path("data/raw/content_refresh_anonymized.csv").exists():
    if not Path("/content/FlyRank-ML").exists():
        !git clone https://github.com/Fatima-05/FlyRank-ML.git /content/FlyRank-ML
    os.chdir("/content/FlyRank-ML")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)

feature_cols = [c for c in [
    "impressions_90d", "clicks_90d", "ctr", "content_age_days",
    "word_count", "avg_position", "search_volume"
] if c in df.columns]
X = df[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
y = df["is_declining"]

def precision_at_k_from_scores(y_true, scores, k=50):
    order = np.argsort(-scores)
    top = y_true.to_numpy()[order][:k]
    return float(top.mean())

# BEFORE: random row split
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X, y, test_size=0.25, random_state=42)
rf_r = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_r.fit(X_tr_r, y_tr_r)
p_random = precision_at_k_from_scores(y_te_r, rf_r.predict_proba(X_te_r)[:, 1])

# AFTER: client-grouped split
groups = df["client_id"] if "client_id" in df.columns else pd.Series(np.arange(len(df)))
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr, te = next(gss.split(X, y, groups))
rf_g = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_g.fit(X.iloc[tr], y.iloc[tr])
p_grouped = precision_at_k_from_scores(y.iloc[te], rf_g.predict_proba(X.iloc[te])[:, 1])

compare = pd.DataFrame({
    "split": ["random_rows", "client_group_holdout"],
    "precision_at_50": [p_random, p_grouped],
})
print(compare)
print("Test base rate (grouped):", round(y.iloc[te].mean(), 3))
compare

                  split  precision_at_50
0           random_rows             0.90
1  client_group_holdout             0.62
Test base rate (grouped): 0.517


,split,precision_at_50
0,random_rows,0.90
1,client_group_holdout,0.62


## 3. Leakage audit

Final feature set must not contain label fields, future outcomes, or identity fields as predictors.

In [7]:
forbidden = [
    "trend_direction", "trend_pct", "is_declining",
    "client_id", "content_id", "client_name", "domain", "url", "query"
]
leaked = [c for c in forbidden if c in feature_cols]
print("Feature cols:", feature_cols)
print("Forbidden cols present in features?", leaked)

# trap: add label into features and show AUC jump
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X_leak = X.copy()
X_leak["is_declining_leaked"] = y.values
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42).split(X, y, groups))

m0 = LogisticRegression(max_iter=500).fit(X.iloc[tr], y.iloc[tr])
m1 = LogisticRegression(max_iter=500).fit(X_leak.iloc[tr], y.iloc[tr])
auc0 = roc_auc_score(y.iloc[te], m0.predict_proba(X.iloc[te])[:, 1])
auc1 = roc_auc_score(y.iloc[te], m1.predict_proba(X_leak.iloc[te])[:, 1])
print(f"AUC clean={auc0:.3f} | AUC with leaked label={auc1:.3f}")

Feature cols: ['impressions_90d', 'clicks_90d', 'ctr', 'content_age_days', 'word_count', 'avg_position', 'search_volume']
Forbidden cols present in features? []
AUC clean=0.558 | AUC with leaked label=1.000


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 4. Claim rewrite

Bold (unsafe):
"The model predicts which pages will recover after a refresh and beats Google-style ranking signals."

Safe rewrite:
"On a client-holdout split of this anonymized internship dataset, a random forest ranked declining pages into the top 50 more often than a hand-written baseline (Precision@50 measured higher for the forest). This is offline decision-support evidence for building a review queue, not a causal claim about refresh impact and not a claim about any search engine's algorithm."

In [8]:
print("Safe language locked: observed / measured / directional / decision-support")

Safe language locked: observed / measured / directional / decision-support


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.